# NVIDIA NIM tool calling with LangGraph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-nvidia/blob/main/cookbook/nvidia_nim_agents.ipynb)

This notebook uses an [NVIDIA Nemotron](https://developer.nvidia.com/nemotron) model served by NVIDIA NIM
to show **tool calling**, one of the core capabilities of agents. As described in this
[introduction to LLM agents](https://developer.nvidia.com/blog/introduction-to-llm-agents/), agents use an LLM
to reason through a problem, plan, and execute that plan with tools.

**Tools** accept structured input, execute an action, and return the result. They often wrap external APIs,
but they do not have to. To keep this notebook self-contained, the tools below return canned data.

## NIM and tool calling

NIM microservices package optimized models (chat, embedding, reranking, and more) behind OpenAI-compatible APIs.
For models trained for function calling, NIM returns structured tool calls that LangChain parses into
`AIMessage.tool_calls`. This notebook uses `nvidia/nemotron-3-super-120b-a12b`.

## Install the packages

In [ ]:
%pip install -qU "langchain-nvidia-ai-endpoints>=1.4,<2" "langchain>=1.0,<2" "langgraph>=1.0,<2"

## Configure the endpoint: hosted or self-hosted

Every model client in this notebook is built from the next cell.

| Variable | Unset (default) | Set |
| --- | --- | --- |
| `NVIDIA_BASE_URL` | NVIDIA-hosted endpoints on [build.nvidia.com](https://build.nvidia.com/) | Your own NIM, e.g. `http://localhost:8000/v1` |
| `NVIDIA_API_KEY` | Required for hosted endpoints (you are prompted if it is missing) | Optional for most self-hosted NIMs |
| `NVIDIA_CHAT_MODEL` | `nvidia/nemotron-3-super-120b-a12b` | The model your NIM serves (see `GET $NVIDIA_BASE_URL/models`) |

In [ ]:
import getpass
import os

# Unset: NVIDIA-hosted endpoints (build.nvidia.com).
# Set, e.g. "http://localhost:8000/v1": a self-hosted NIM.
NVIDIA_BASE_URL = os.environ.get("NVIDIA_BASE_URL") or None
CHAT_MODEL = os.environ.get("NVIDIA_CHAT_MODEL", "nvidia/nemotron-3-super-120b-a12b")

if NVIDIA_BASE_URL:
    print(f"Using self-hosted NIM at {NVIDIA_BASE_URL}")
else:
    print("Using NVIDIA-hosted endpoints (build.nvidia.com)")
    if not os.environ.get("NVIDIA_API_KEY"):
        os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA API key (nvapi-...): ")

# Shared keyword arguments for every NVIDIA client in this notebook.
# ChatNVIDIA reads NVIDIA_API_KEY from the environment.
NIM_KWARGS = {"base_url": NVIDIA_BASE_URL} if NVIDIA_BASE_URL else {}
print(f"Chat model: {CHAT_MODEL}")

Declare a model that supports tool calling.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(model=CHAT_MODEL, temperature=0, **NIM_KWARGS)

## Define tools

In [ ]:
from langchain_core.tools import tool

# Offline stand-ins for real services, so no third-party API keys are needed.
_FAKE_WEATHER = {
    "santa clara": "22°C and sunny",
    "san francisco": "16°C with fog",
    "new york": "9°C and raining",
}


@tool
def get_current_location() -> str:
    """Return the user's current city."""
    return "Santa Clara"


@tool
def get_weather(city: str) -> str:
    """Return the current weather for a city."""
    return _FAKE_WEATHER.get(city.lower(), f"No weather data for {city}")


tools = [get_current_location, get_weather]

Invoke a tool manually to see the result.

In [ ]:
get_weather.invoke({"city": "San Francisco"})

Bind the tools to the chat model so it can decide when to call them.

In [ ]:
llm_with_tools = llm.bind_tools(tools)
llm_with_tools.invoke("What's the weather in San Francisco?").tool_calls

The model returns a `get_weather` tool call with structured arguments instead of a text answer.
Here is the end-to-end process of calling and invoking a tool:

In [ ]:
ai_response = llm_with_tools.invoke("What's the weather in San Francisco?")
tools_by_name = {t.name: t for t in tools}
for tool_call in ai_response.tool_calls:
    result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
    print(f"Tool: {tool_call['name']}({tool_call['args']}) -> {result}")

## A tool-calling loop with LangGraph

Questions like "What's the weather where I am?" need several tool calls in sequence. LangGraph models this
loop as a graph:

1. **State:** the shared data, here a list of messages (`MessagesState`).
2. **Nodes:** functions that update the state, here the model (`agent`) and a prebuilt `ToolNode`.
3. **Edges:** the routing logic. `tools_condition` goes to `tools` while the model requests tool calls, and ends otherwise.

In [ ]:
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition


def call_model(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


workflow = StateGraph(MessagesState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(tools))
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", tools_condition)
workflow.add_edge("tools", "agent")

app = workflow.compile()

The agent keeps calling tools until it has what it needs.

In [ ]:
try:
    from IPython.display import Image, display

    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    # Rendering the PNG needs network access to mermaid.ink.
    print(app.get_graph().draw_mermaid())

Run a query that needs one tool call, then one that needs two.

In [ ]:
for chunk in app.stream(
    {"messages": [("human", "What's the weather in San Francisco?")]},
    stream_mode="values",
):
    chunk["messages"][-1].pretty_print()

In [ ]:
for chunk in app.stream(
    {"messages": [("human", "What's the weather where I currently am?")]},
    stream_mode="values",
):
    chunk["messages"][-1].pretty_print()

## Next steps

- LangChain v1 packages this loop as `langchain.agents.create_agent`; see
  [nemotron_tool_calling_agent.ipynb](https://github.com/langchain-ai/langchain-nvidia/blob/main/cookbook/nemotron_tool_calling_agent.ipynb).
- For planning, a virtual filesystem, and subagents, see
  [nemotron_deep_agent_nim.ipynb](https://github.com/langchain-ai/langchain-nvidia/blob/main/cookbook/nemotron_deep_agent_nim.ipynb).
- To swap the offline tools for live web search, add a search tool such as
  [Tavily](https://docs.langchain.com/oss/python/integrations/tools/tavily_search) to `tools`.

## Related

- [Cookbook index](https://github.com/langchain-ai/langchain-nvidia/tree/main/cookbook)
- [`langchain-nvidia-ai-endpoints` README](https://github.com/langchain-ai/langchain-nvidia/blob/main/libs/ai-endpoints/README.md)
- [Model registry: current and deprecated model IDs](https://github.com/langchain-ai/langchain-nvidia/blob/main/libs/ai-endpoints/docs/model_registry.md)
- [NVIDIA NIM for LLMs documentation](https://docs.nvidia.com/nim/large-language-models/latest/introduction.html)